# Model Training

Optuna hyperparameter tuning and model selection pipeline.

In [1]:
"""
Atmos â€” Machine Learning Engine (Phase 2)

"""

import pandas as pd
import numpy as np
import os
import joblib
import optuna
import lightgbm as lgb
import xgboost as xgb
from sklearn.metrics import root_mean_squared_error

# Target configuration
TARGET = "pm25"

def load_and_split_data():
    path = os.path.join("../data_store", "cache", "features_delhi.csv")
    print(f"Loading data from {path}...")
    df = pd.read_csv(path)
    
    # Sort by time to prevent data leakage
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.sort_values(by="timestamp").reset_index(drop=True)
    
    # We want to forecast 24 hours into the future
    print("Creating 24h future target...")
    df["target_24h"] = df.groupby("station")[TARGET].shift(-24)
    df = df.dropna(subset=["target_24h"]).reset_index(drop=True)
    
    features = [c for c in df.columns if c not in ["timestamp", "station", "ward_id", "target_24h", "aqi"]]
    
    # Time-based split: Train on first 80% of time, Test on last 20%
    split_idx = int(len(df) * 0.8)
    train_df = df.iloc[:split_idx]
    # Purge gap: drop 24 hours between train and test to prevent target leakage
    # Assuming hourly data, 24 rows = 24 hours per station.
    # Actually, data is sorted by time across ALL stations. 
    # Let's drop rows where timestamp is within 24 hours of the last train timestamp.
    train_end_time = train_df['timestamp'].max()
    purge_end_time = train_end_time + pd.Timedelta(hours=24)
    test_df = df[df['timestamp'] > purge_end_time]
    
    X_train, y_train = train_df[features], train_df["target_24h"]
    X_test, y_test = test_df[features], test_df["target_24h"]
    
    return X_train, y_train, X_test, y_test, features, df

def objective_lgb(trial, X_train, y_train, X_test, y_test):
    params = {
        "objective": "regression",
        "metric": "rmse",
        "verbosity": -1,
        "boosting_type": "gbdt",
        "n_estimators": trial.suggest_int("n_estimators", 50, 200),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 20, 100),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.6, 1.0),
    }
    model = lgb.LGBMRegressor(**params, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    rmse = root_mean_squared_error(y_test, preds)
    return rmse

def objective_xgb(trial, X_train, y_train, X_test, y_test):
    params = {
        "objective": "reg:squarederror",
        "n_estimators": trial.suggest_int("n_estimators", 50, 200),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
    }
    model = xgb.XGBRegressor(**params, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    rmse = root_mean_squared_error(y_test, preds)
    return rmse

def run_pipeline():
    X_train, y_train, X_test, y_test, features, full_df = load_and_split_data()
    
    # Subsample training data for rapid prototyping
    print("Tuning LightGBM...")
    study_lgb = optuna.create_study(direction="minimize")
    study_lgb.optimize(lambda t: objective_lgb(t, X_train, y_train, X_test, y_test), n_trials=100)
    
    print("Tuning XGBoost...")
    study_xgb = optuna.create_study(direction="minimize")
    study_xgb.optimize(lambda t: objective_xgb(t, X_train, y_train, X_test, y_test), n_trials=100)
    
    best_lgb_rmse = study_lgb.best_value
    best_xgb_rmse = study_xgb.best_value
    
    print("Optuna Results:")
    print(f"Best LightGBM RMSE: {best_lgb_rmse:.2f}")
    print(f"Best XGBoost RMSE:  {best_xgb_rmse:.2f}")
    
    # Select Champion
    champion = "LightGBM" if best_lgb_rmse < best_xgb_rmse else "XGBoost"
    print(f"Champion Model: {champion}")
    
    # Train the champion on the FULL dataset using Quantile Regression!
    X_full = full_df[features]
    y_full = full_df["target_24h"]
    
    out_dir = os.path.join("../data_store", "models")
    os.makedirs(out_dir, exist_ok=True)
    
    print("Training quantile regression models...")
    
    if champion == "LightGBM":
        params = study_lgb.best_params
        for q in [0.1, 0.5, 0.9]:
            print(f"  Training {champion} (Quantile {q})...")
            # Override objective for quantile
            model = lgb.LGBMRegressor(**params, objective="quantile", alpha=q, random_state=42, n_jobs=-1)
            model.fit(X_full, y_full)
            path = os.path.join(out_dir, f"champion_q{int(q*100)}.pkl")
            joblib.dump(model, path)
            print(f"  Saved to {path}")
            
    else:
        params = study_xgb.best_params
        for q in [0.1, 0.5, 0.9]:
            print(f"  Training {champion} (Quantile {q})...")
            # XGBoost uses 'reg:quantileerror' in recent versions with quantile_alpha
            model = xgb.XGBRegressor(**params, objective="reg:quantileerror", quantile_alpha=q, random_state=42, n_jobs=-1)
            model.fit(X_full, y_full)
            path = os.path.join(out_dir, f"champion_q{int(q*100)}.pkl")
            joblib.dump(model, path)
            print(f"  Saved to {path}")
            
    # Save the feature names so the API knows the exact expected order
    joblib.dump(features, os.path.join(out_dir, "feature_names.pkl"))
    print("Training complete.")

if __name__ == "__main__":
    optuna.logging.set_verbosity(optuna.logging.WARNING) # Suppress noisy logs
    run_pipeline()


/Users/Admin/Downloads/atmos/backend/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading data from ../data_store/cache/features_delhi.csv...


Creating 24h future target...
Tuning LightGBM...


Tuning XGBoost...


Optuna Results:
Best LightGBM RMSE: 29.90
Best XGBoost RMSE:  29.93
Champion Model: LightGBM
Training quantile regression models...
  Training LightGBM (Quantile 0.1)...


  Saved to ../data_store/models/champion_q10.pkl
  Training LightGBM (Quantile 0.5)...


  Saved to ../data_store/models/champion_q50.pkl
  Training LightGBM (Quantile 0.9)...


  Saved to ../data_store/models/champion_q90.pkl
Training complete.
